# 95 FS3 XGBoost Tuning Workbench

This notebook is a **validation-only tuning workbench** for the active `FS3` XGBoost survivor.

Current scope:
- model family: `XGBoost`
- fixed feature bundle: accepted `FS3` pruned candidate
- primary objective: improve validation `stitched_all_horizon` MAE
- secondary checks: keep coverage acceptable and inspect `d_only` / `guidance_only` as diagnostics
- hard rule: do **not** use the final test set during tuning

This notebook is **not** part of the fixed benchmark sequence. It exists to make the `FS3` retuning step traceable and methodologically explicit before any final benchmark rerun is frozen.

In [ ]:
import json
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.reporting import find_latest_run, load_csv
from hourly_da.core.tuning import build_tuning_placeholder, tuning_snippet_frame
from hourly_da.core.tuning_workbench import (
    XGBOOST_FS2_PARENT_RUN_LABEL,
    XGBOOST_FS3_ACTIVE_EXCLUDED_BLOCKS,
    XGBOOST_FS3_ACTIVE_MODEL_NAME,
    XGBOOST_FS3_ACTIVE_PARENT_RUN_LABEL,
    XGBOOST_FS3_PREVIOUS_PARENT_RUN_LABEL,
    build_origin_schedule_by_split,
    build_tuning_recommendation,
    build_tuning_candidate_summary,
    build_xgboost_fs3_tuning_baselines,
    build_xgboost_fs3_tuning_candidate_models,
    list_saved_run_dirs,
    origin_schedule_summary_frame,
    render_tuning_recommendation_markdown,
    resolve_xgboost_fs3_tuning_parent,
    with_selected_evaluation_splits,
)
from hourly_da.notebook_support import (
    apply_notebook_display_defaults,
    estimate_run_duration_seconds,
    format_duration,
    run_suite_with_feedback,
)

apply_notebook_display_defaults()

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root
analysis_root = output_root / "analysis" / "xgboost_fs3_tuning"
analysis_root.mkdir(parents=True, exist_ok=True)


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


## What tuning means here

In this notebook, tuning means **changing hyperparameters while keeping the accepted `FS3` feature bundle fixed**.

That matters because two different questions should not be mixed:
1. feature redesign: which `FS3` families or blocks belong in the model at all
2. hyperparameter tuning: how aggressively the fixed `FS3` model should fit that frozen structure

The accepted XGBoost `FS3` candidate already answers the first question. This notebook addresses only the second.

The tuning flow is deliberately two-stage:
1. **Pilot screen** on a smaller but seasonally distributed subset of validation origins
2. **Full validation confirmation** on the shortlisted candidates only

The pilot is allowed to be approximate because it is only a screening device. The final choice must come from the full validation rerun.

In [ ]:
display(pd.DataFrame([build_tuning_placeholder("xgboost", "FS3")]))
display(tuning_snippet_frame(model_family="xgboost", fs_level="FS3"))

saved_runs = list_saved_run_dirs(output_root, "xgboost_fs3_tuning_")
if saved_runs.empty:
    print("No saved tuning runs were found yet for the 'xgboost_fs3_tuning_' prefix.")
else:
    display(saved_runs)


In [ ]:
parent = resolve_xgboost_fs3_tuning_parent(config)
d_settings = dict(parent.settings_payload.get("d_only_model_settings") or {})
g_settings = dict(parent.settings_payload.get("guidance_model_settings") or {})

baseline_context = pd.DataFrame([
    {
        "active_parent_run_label": XGBOOST_FS3_ACTIVE_PARENT_RUN_LABEL,
        "previous_parent_run_label": XGBOOST_FS3_PREVIOUS_PARENT_RUN_LABEL,
        "fs2_reference_run_label": XGBOOST_FS2_PARENT_RUN_LABEL,
        "model_name_inside_run": parent.model_name,
        "d_only_experiment_code": str((d_settings.get("fs3_experiment") or {}).get("code", "")),
        "guidance_experiment_code": str((g_settings.get("fs3_experiment") or {}).get("code", "")),
        "excluded_blocks": ", ".join(XGBOOST_FS3_ACTIVE_EXCLUDED_BLOCKS),
        "learning_rate": d_settings.get("learning_rate"),
        "max_depth": d_settings.get("max_depth"),
        "n_estimators": d_settings.get("n_estimators"),
        "subsample": d_settings.get("subsample"),
        "colsample_bytree": d_settings.get("colsample_bytree"),
        "reg_alpha": d_settings.get("reg_alpha"),
        "reg_lambda": d_settings.get("reg_lambda"),
        "training_window_hours": d_settings.get("training_window_hours"),
        "min_train_rows": d_settings.get("min_train_rows"),
    }
])
display(baseline_context.T.rename(columns={0: "value"}))

baseline_run_dir = latest_run_or_none(XGBOOST_FS3_ACTIVE_PARENT_RUN_LABEL)
if baseline_run_dir is not None:
    baseline_summary = build_tuning_candidate_summary(
        load_csv(baseline_run_dir, "metrics_by_reporting_level.csv"),
        load_csv(baseline_run_dir, "origin_timing_summary.csv"),
        split_name="validation",
        baseline_model=XGBOOST_FS3_ACTIVE_MODEL_NAME,
    )
    display(
        baseline_summary[
            [
                "model",
                "mae_d_only",
                "mae_guidance_only",
                "mae_stitched_all_horizon",
                "coverage_stitched_all_horizon_pct",
                "fit_time_mean_sec",
            ]
        ]
    )


## Search protocol

The search is intentionally compact.

Decision rules:
1. Keep the accepted `FS3` bundle fixed. Do not mix feature pruning with hyperparameter search.
2. Use **validation only**. The test set stays untouched until one final configuration is frozen.
3. Use validation `stitched_all_horizon` MAE as the primary selection metric.
4. Use `d_only` and `guidance_only` as interpretation slices, not separate primary objectives.
5. Reject candidates that create clear coverage problems or fit-time warnings.
6. If gains are very small, prefer the simpler or faster configuration rather than over-claiming a win.

The first pass below tunes the tree region around the current settings. If that region does not move materially, a later second pass can target training-window length around the winning tree region instead of widening everything at once.

In [ ]:
RUN_LABEL_PREFIX = "xgboost_fs3_tuning"
PILOT_RUN_LABEL = f"{RUN_LABEL_PREFIX}_pilot_v1"
FULL_VALIDATION_RUN_LABEL = f"{RUN_LABEL_PREFIX}_full_validation_v1"

PRIMARY_SPLIT = "validation"
PRIMARY_REPORTING_LEVEL = "stitched_all_horizon"
PRIMARY_METRIC = "mae"
MIN_VALIDATION_COVERAGE_PCT = 95.0
PILOT_ORIGIN_COUNT = 48
AUTO_FULL_VALIDATION_TOP_K = 3
FULL_VALIDATION_OVERRIDE_IDS: list[str] = []

candidate_plan = pd.DataFrame(
    [
        {
            "candidate_id": "xgboost_fs3_tune_r1_c01",
            "search_stage": "pilot_round_1",
            "learning_rate": 0.03,
            "max_depth": 4,
            "n_estimators": 140,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0,
            "rationale": "Lower learning rate with more trees around the current depth.",
        },
        {
            "candidate_id": "xgboost_fs3_tune_r1_c02",
            "search_stage": "pilot_round_1",
            "learning_rate": 0.03,
            "max_depth": 3,
            "n_estimators": 180,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0,
            "rationale": "Shallower trees with more boosting rounds for smoother generalization.",
        },
        {
            "candidate_id": "xgboost_fs3_tune_r1_c03",
            "search_stage": "pilot_round_1",
            "learning_rate": 0.05,
            "max_depth": 5,
            "n_estimators": 120,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0,
            "rationale": "Adds capacity while keeping the original learning rate.",
        },
        {
            "candidate_id": "xgboost_fs3_tune_r1_c04",
            "search_stage": "pilot_round_1",
            "learning_rate": 0.05,
            "max_depth": 3,
            "n_estimators": 100,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0,
            "rationale": "Simpler tree shape to test whether the current depth is too aggressive.",
        },
        {
            "candidate_id": "xgboost_fs3_tune_r1_c05",
            "search_stage": "pilot_round_1",
            "learning_rate": 0.07,
            "max_depth": 4,
            "n_estimators": 90,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0,
            "rationale": "Tests a mildly more aggressive shrinkage region with limited extra depth.",
        },
        {
            "candidate_id": "xgboost_fs3_tune_r1_c06",
            "search_stage": "pilot_round_1",
            "learning_rate": 0.03,
            "max_depth": 4,
            "n_estimators": 140,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_alpha": 0.0,
            "reg_lambda": 2.0,
            "rationale": "Same conservative tree region as c01 but with stronger L2 regularization.",
        },
        {
            "candidate_id": "xgboost_fs3_tune_r1_c07",
            "search_stage": "pilot_round_1",
            "learning_rate": 0.03,
            "max_depth": 4,
            "n_estimators": 140,
            "subsample": 0.9,
            "colsample_bytree": 0.9,
            "reg_alpha": 0.2,
            "reg_lambda": 1.0,
            "rationale": "Adds sparse regularization on top of the conservative c01 region.",
        },
        {
            "candidate_id": "xgboost_fs3_tune_r1_c08",
            "search_stage": "pilot_round_1",
            "learning_rate": 0.03,
            "max_depth": 4,
            "n_estimators": 140,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0,
            "rationale": "Adds stochastic row and column subsampling to reduce variance.",
        },
    ]
)
candidate_plan.to_csv(analysis_root / "pilot_candidate_plan.csv", index=False)
display(candidate_plan)


In [ ]:
validation_only_config = with_selected_evaluation_splits(config, (PRIMARY_SPLIT,))
full_validation_schedule = build_origin_schedule_by_split(
    validation_only_config,
    split_names=(PRIMARY_SPLIT,),
)
pilot_validation_schedule = build_origin_schedule_by_split(
    validation_only_config,
    split_names=(PRIMARY_SPLIT,),
    max_origins_by_split={PRIMARY_SPLIT: PILOT_ORIGIN_COUNT},
)

schedule_summary = pd.concat(
    [
        origin_schedule_summary_frame(
            validation_only_config,
            split_name=PRIMARY_SPLIT,
            sampled_schedule=pilot_validation_schedule[PRIMARY_SPLIT],
        ),
        origin_schedule_summary_frame(
            validation_only_config,
            split_name=PRIMARY_SPLIT,
            sampled_schedule=full_validation_schedule[PRIMARY_SPLIT],
        ).assign(note="full_validation_reference"),
    ],
    ignore_index=True,
)
display(schedule_summary)

estimate = estimate_run_duration_seconds(output_root, XGBOOST_FS3_ACTIVE_PARENT_RUN_LABEL)
if estimate is not None:
    base_timing = load_csv(estimate["run_dir"], "origin_timing_summary.csv")
    xgb_rows = base_timing[base_timing["model"].astype(str) == XGBOOST_FS3_ACTIVE_MODEL_NAME].copy()
    if not xgb_rows.empty:
        xgb_seconds_per_origin = float((xgb_rows["fit_time_mean_sec"] + xgb_rows["predict_time_mean_sec"]).mean())
        projected_pilot_seconds = xgb_seconds_per_origin * pilot_validation_schedule[PRIMARY_SPLIT].shape[0] * (candidate_plan.shape[0] + 2)
        projected_full_seconds = xgb_seconds_per_origin * full_validation_schedule[PRIMARY_SPLIT].shape[0] * (AUTO_FULL_VALIDATION_TOP_K + 2)
        runtime_frame = pd.DataFrame(
            [
                {
                    "stage": "pilot_screen",
                    "origins": pilot_validation_schedule[PRIMARY_SPLIT].shape[0],
                    "candidate_models": candidate_plan.shape[0],
                    "rough_estimated_runtime": format_duration(projected_pilot_seconds),
                },
                {
                    "stage": "full_validation_confirmation",
                    "origins": full_validation_schedule[PRIMARY_SPLIT].shape[0],
                    "candidate_models": AUTO_FULL_VALIDATION_TOP_K,
                    "rough_estimated_runtime": format_duration(projected_full_seconds),
                },
            ]
        )
        display(runtime_frame)


## Optional pilot execution hook

The pilot run keeps the methodology strict in one sense and relaxed in another:
- strict: it still uses the shared rolling-origin benchmark machinery and validation only
- relaxed: it uses only a subset of validation origins so the first search round stays tractable

Interpretation rule for the pilot:
- do **not** declare a final winner from the pilot
- do use the pilot to reject obviously weak regions and to promote only a small shortlist into full validation

In [ ]:
ALLOW_PILOT_RERUN = False

pilot_models = [
    *build_xgboost_fs3_tuning_baselines(
        validation_only_config,
        current_baseline_name="xgboost_fs3_current_baseline",
    ),
    *build_xgboost_fs3_tuning_candidate_models(validation_only_config, candidate_plan),
]

if ALLOW_PILOT_RERUN:
    pilot_result = run_suite_with_feedback(
        validation_only_config,
        run_label=PILOT_RUN_LABEL,
        models=pilot_models,
        include_external_features=True,
        show_progress=True,
        progress_label=PILOT_RUN_LABEL,
        origin_schedule_by_split=pilot_validation_schedule,
    )
else:
    print("Pilot rerun is disabled. Set ALLOW_PILOT_RERUN = True to execute the validation-only pilot screen.")


In [ ]:
pilot_run_dir = latest_run_or_none(PILOT_RUN_LABEL)
pilot_summary = pd.DataFrame()
auto_full_validation_ids: list[str] = []
model_order = [
    "naive_previous_week",
    "naive_previous_year",
    "xgboost_fs2_anchor",
    "xgboost_fs3_current_baseline",
    *candidate_plan["candidate_id"].astype(str).tolist(),
]

if pilot_run_dir is None:
    print("No saved pilot run found yet.")
else:
    pilot_summary = build_tuning_candidate_summary(
        load_csv(pilot_run_dir, "metrics_by_reporting_level.csv"),
        load_csv(pilot_run_dir, "origin_timing_summary.csv"),
        split_name=PRIMARY_SPLIT,
        baseline_model="xgboost_fs3_current_baseline",
        model_order=model_order,
    )
    pilot_summary.to_csv(analysis_root / "pilot_summary.csv", index=False)
    display(
        pilot_summary[
            [
                "model",
                "mae_d_only",
                "mae_guidance_only",
                "mae_stitched_all_horizon",
                "delta_vs_baseline_stitched_mae",
                "coverage_stitched_all_horizon_pct",
                "fit_time_mean_sec",
                "fit_time_warning_count",
            ]
        ]
    )

    candidate_only = pilot_summary[pilot_summary["model"].isin(candidate_plan["candidate_id"].astype(str))].copy()
    auto_promotions = candidate_only[
        (candidate_only["delta_vs_baseline_stitched_mae"] < 0.0)
        & (candidate_only["coverage_stitched_all_horizon_pct"] >= MIN_VALIDATION_COVERAGE_PCT)
        & (candidate_only["fit_time_warning_count"].fillna(0) == 0)
    ].sort_values(
        ["delta_vs_baseline_stitched_mae", "mae_d_only", "fit_time_mean_sec"],
        ascending=[True, True, True],
    ).head(AUTO_FULL_VALIDATION_TOP_K)
    auto_full_validation_ids = auto_promotions["model"].astype(str).tolist()

    if auto_promotions.empty:
        print("No pilot candidate met the automatic promotion rule yet.")
    else:
        display(
            auto_promotions[
                [
                    "model",
                    "mae_stitched_all_horizon",
                    "delta_vs_baseline_stitched_mae",
                    "mae_d_only",
                    "mae_guidance_only",
                    "fit_time_mean_sec",
                ]
            ]
        )


## How to read the pilot output

Good pilot behavior looks like this:
- `stitched_all_horizon` MAE improves versus `xgboost_fs3_current_baseline`
- coverage stays near the benchmark standard
- `d_only` is not collapsing even if it moves slightly
- fit time does not blow up without a compensating accuracy gain

Bad pilot behavior looks like this:
- full-horizon MAE gets worse
- coverage falls below the acceptable floor
- fit warnings appear
- only one narrow slice improves while the main objective clearly deteriorates

The pilot should therefore shrink the search space, not finalize the thesis conclusion.

In [ ]:
FULL_VALIDATION_CANDIDATE_IDS = FULL_VALIDATION_OVERRIDE_IDS or auto_full_validation_ids
ALLOW_FULL_VALIDATION_RERUN = False

if not FULL_VALIDATION_CANDIDATE_IDS:
    print("No full-validation candidates are selected yet. Run the pilot first or set FULL_VALIDATION_OVERRIDE_IDS manually.")
else:
    full_validation_specs = candidate_plan[candidate_plan["candidate_id"].isin(FULL_VALIDATION_CANDIDATE_IDS)].copy()
    display(full_validation_specs)

    if ALLOW_FULL_VALIDATION_RERUN:
        full_models = [
            *build_xgboost_fs3_tuning_baselines(
                validation_only_config,
                current_baseline_name="xgboost_fs3_current_baseline",
            ),
            *build_xgboost_fs3_tuning_candidate_models(validation_only_config, full_validation_specs),
        ]
        full_validation_result = run_suite_with_feedback(
            validation_only_config,
            run_label=FULL_VALIDATION_RUN_LABEL,
            models=full_models,
            include_external_features=True,
            show_progress=True,
            progress_label=FULL_VALIDATION_RUN_LABEL,
            origin_schedule_by_split=full_validation_schedule,
        )
    else:
        print("Full-validation rerun is disabled. Set ALLOW_FULL_VALIDATION_RERUN = True once the pilot shortlist is frozen.")


In [ ]:
full_validation_run_dir = latest_run_or_none(FULL_VALIDATION_RUN_LABEL)
full_validation_summary = pd.DataFrame()
winner = pd.DataFrame()

if full_validation_run_dir is None:
    print("No saved full-validation run found yet.")
else:
    full_validation_summary = build_tuning_candidate_summary(
        load_csv(full_validation_run_dir, "metrics_by_reporting_level.csv"),
        load_csv(full_validation_run_dir, "origin_timing_summary.csv"),
        split_name=PRIMARY_SPLIT,
        baseline_model="xgboost_fs3_current_baseline",
        model_order=model_order,
    )
    full_validation_summary.to_csv(analysis_root / "full_validation_summary.csv", index=False)
    display(
        full_validation_summary[
            [
                "model",
                "mae_d_only",
                "mae_guidance_only",
                "mae_stitched_all_horizon",
                "delta_vs_baseline_stitched_mae",
                "coverage_stitched_all_horizon_pct",
                "fit_time_mean_sec",
                "fit_time_warning_count",
            ]
        ]
    )

    finalist_rows = full_validation_summary[full_validation_summary["model"].isin(FULL_VALIDATION_CANDIDATE_IDS)].copy()
    if finalist_rows.empty:
        print("No finalist rows were found for the current full-validation candidate ids.")
    else:
        winner = finalist_rows.sort_values(
            ["mae_stitched_all_horizon", "mae_d_only", "fit_time_mean_sec"],
            ascending=[True, True, True],
        ).head(1)
        display(
            winner[
                [
                    "model",
                    "mae_stitched_all_horizon",
                    "delta_vs_baseline_stitched_mae",
                    "mae_d_only",
                    "mae_guidance_only",
                    "fit_time_mean_sec",
                ]
            ]
        )


In [ ]:
recommendation_payload: dict[str, object] = {}
recommendation_json_path = analysis_root / "recommended_settings.json"
recommendation_md_path = analysis_root / "recommendation_summary.md"

if full_validation_summary.empty:
    print("No full-validation summary available yet. Recommendation files have not been written.")
else:
    recommendation_payload = build_tuning_recommendation(
        candidate_plan,
        full_validation_summary,
        finalist_candidate_ids=FULL_VALIDATION_CANDIDATE_IDS,
        baseline_model="xgboost_fs3_current_baseline",
        min_coverage_pct=MIN_VALIDATION_COVERAGE_PCT,
    )
    recommendation_json_path.write_text(json.dumps(recommendation_payload, indent=2), encoding="utf-8")
    recommendation_md_path.write_text(render_tuning_recommendation_markdown(recommendation_payload), encoding="utf-8")

    if recommendation_payload.get("recommendation"):
        display(pd.DataFrame([recommendation_payload["recommendation"]]))
    if recommendation_payload.get("recommended_settings"):
        display(pd.DataFrame([recommendation_payload["recommended_settings"]]))

    ranking_frame = pd.DataFrame(recommendation_payload.get("finalist_ranking", []))
    if not ranking_frame.empty:
        display(ranking_frame)

    print(f"Recommendation JSON: {recommendation_json_path}")
    print(f"Recommendation Markdown: {recommendation_md_path}")


## Final interpretation rule

Once the full validation run exists, the conclusion should be written in this order:
1. **Primary claim**: which candidate has the best validation `stitched_all_horizon` MAE
2. **Diagnostic reading**: whether `d_only` and `guidance_only` support or complicate that result
3. **Practical reading**: whether the gain is large enough to justify the runtime and complexity change
4. **Freeze step**: lock the winning `FS3` settings and rerun the standard benchmark pipeline only after the choice is fixed

Important interpretation note:
- if the best candidate is only trivially better than the current baseline, keeping the current baseline is a defensible conclusion
- if a candidate improves only one slice but worsens the main full-horizon objective, it should not be promoted
- the test set should be consulted only after the validation winner is already frozen
- the notebook writes `recommended_settings.json` and `recommendation_summary.md`, but it does **not** freeze settings automatically

Expected follow-up after a winner is frozen:
- rerun the official benchmark context with the chosen `FS3` settings
- compare the frozen winner against the saved LEAR benchmark and the official naive benchmark
- report the final test result as holdout confirmation, not as a tuning input